# 02 - H2 Transfer statt Aufbau

**Der Bund verteilt zunehmend Investitionsmittel an Länder und Private, baut aber selbst nicht zunehmend auf.**

Essay-Bezug: Akt 2 (inklusive EP-60-Spin-off und Methoden-Box Verpflichtungsermächtigungen)

## These und politischer Anschluss

**Original-These.** Der konsumtive Anteil am Digital-Soll steigt zwischen 2019 und 2024 um mindestens 5 Prozentpunkte gegenüber dem investiven, Ausdruck einer wiederkehrenden Sorge in der Digitalpolitik: Der Bund finanziere zunehmend den Betrieb bestehender IT-Systeme statt Transformation.

**Präzisierte These nach erster Sichtung.** Die Daten widersprechen der Erwartung zunächst, der Investitionsanteil springt 2024 sichtbar nach oben. Bei genauerer Betrachtung verschwindet dieser Sprung als eigenständiges Phänomen: Er ist Symptom einer anderen, stärkeren Verschiebung. Eigene Bundes-Investitionen stagnieren, Investitions-Zuschüsse an Länder und Private wachsen kräftig. Der Bund verteilt zunehmend, baut aber nicht selbst auf.

**Was wir prüfen.** Erstens das Verhältnis von konsumtiven zu investiven Ausgaben über die Zeit. Zweitens, ob der Investitions-Sprung von eigenen Bundes-Investitionen (Obergruppen 81/82) oder von Investitions-Zuschüssen an Dritte (Obergruppen 83/86/87/88/89) getrieben wird. Drittens die Treiber auf Einzelplan-Ebene. Viertens die Robustheit unter der weiten Abgrenzung. Fünftens einen Spin-off: Was steckt hinter dem auffälligen HG-6-Peak 2023?

## Methodik dieses Drilldowns

**Daten und Abgrenzung.** `digi_soll_eng`, gefiltert auf digital getaggte Titel mit positivem Digital-Anteil (`any_tag == 1 & digi_soll_eng > 0`). Die Hauptgruppen-Logik der Bundeshaushaltsordnung trennt konsumtiv (HG 4–6) von investiv (HG 7–8); innerhalb HG 8 trennt die Obergruppe eigene Bundes-Investitionen (`A_eigene_Invest`, UG 81/82) von Investitions-Zuschüssen an Dritte (`B_Transfer_Invest`, UG 83/86/87/88/89). Mapping in `src.load.UG_INVEST_TYP`.

**Pre-Mortem.** Was würde die Transfer-These töten? Wenn der Investitions-Sprung 2024 vor allem aus UG 81/82 käme, dann wäre es tatsächlich eine Investitionsoffensive des Bundes selbst, keine Verteil-Politik. Auch eine fehlende Robustheit unter `digi_soll_weit` würde die These schwächen.

**Erwartung.** Eigene Investitionen stagnieren absolut und schrumpfen anteilig; Investitions-Zuschüsse an Dritte wachsen deutlich stärker und treiben den beobachteten Sprung, konzentriert auf wenige große Förderprogramme (Breitband, DigitalPakt).

In [1]:
"""02 - H2: Transfer statt Aufbau - Hauptgruppen-Drilldown.

Hauptbefund: Eigene Bundes-Investitionen (UG 81/82) stagnieren. Investitions-
zuschuesse an Dritte (UG 83/86/87/88/89) verdoppeln ihren Anteil.

Drilldowns:
  1. Konsumtiv vs. Investiv ueber Jahre
  2. Investiv-Detail: eigene vs. Transfer
  3. Treiber-Identifikation (welche Einzelplaene, welche Titel)
  4. Robustheit weite Abgrenzung
  5. HG-6-Spin-off: Mikroelektronik-Peak 2023

Outputs: figures/h2_schere_eigene_vs_transfer.png
"""

'02 - H2: Transfer statt Aufbau - Hauptgruppen-Drilldown.\n\nHauptbefund: Eigene Bundes-Investitionen (UG 81/82) stagnieren. Investitions-\nzuschuesse an Dritte (UG 83/86/87/88/89) verdoppeln ihren Anteil.\n\nDrilldowns:\n  1. Konsumtiv vs. Investiv ueber Jahre\n  2. Investiv-Detail: eigene vs. Transfer\n  3. Treiber-Identifikation (welche Einzelplaene, welche Titel)\n  4. Robustheit weite Abgrenzung\n  5. HG-6-Spin-off: Mikroelektronik-Peak 2023\n\nOutputs: figures/h2_schere_eigene_vs_transfer.png\n'

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, '..')  # repo root, relativ zum notebooks/-Verzeichnis

import pandas as pd
from src.load import load, KATS

df = load()  # nutzt Repo-Root-Default aus src.load

# Nur digitale Titel mit eng>0
d = df[(df['any_tag'] == 1) & (df['digi_soll_eng'] > 0)].copy()

In [3]:
print("=== Konsumtiv (HG 4-6) vs. Investiv (HG 7-8) - digi_soll_eng (Mrd) ===")
d['typ_kons_inv'] = d['hg'].map(
    lambda x: 'konsumtiv' if x in ['4', '5', '6']
    else ('investiv' if x in ['7', '8'] else 'sonstiges')
)
agg = d.groupby(['jahr', 'typ_kons_inv'])['digi_soll_eng'].sum().div(1e6).unstack()
agg['Sigma'] = agg.sum(axis=1)
for c in ['konsumtiv', 'investiv', 'sonstiges']:
    if c in agg.columns:
        agg[f'{c}_%'] = (agg[c] / agg['Sigma'] * 100).round(1)
print(agg.round(2))

=== Konsumtiv (HG 4-6) vs. Investiv (HG 7-8) - digi_soll_eng (Mrd) ===
typ_kons_inv  investiv  konsumtiv  Sigma  konsumtiv_%  investiv_%
jahr                                                             
2019              2.32       6.18   8.51         72.7        27.3
2021              4.34      12.21  16.55         73.8        26.2
2023              5.03      14.16  19.19         73.8        26.2
2024              6.85      11.09  17.94         61.8        38.2


**Befund.** Der investive Anteil bleibt 2019–2023 stabil bei rund einem Viertel und springt 2024 sichtbar nach oben, auf den ersten Blick eine Investitionsoffensive. Aber ein Sprung in der Aggregatzahl sagt noch nichts darüber, welche Art von Investition wächst.

Nächster Schritt: Treibt eigene Bundes-Investition (UG 81/82) den Sprung, oder Investitions-Zuschüsse an Dritte (UG 83/86/87/88/89)?

In [4]:
#| label: tbl-schere
print("\n=== HG 8 Detail: eigene vs. Transfer-Investitionen (Mrd) ===")
hg8 = d[d['hg'] == '8']
piv = hg8.groupby(['ug', 'jahr'])['digi_soll_eng'].sum().div(1e6).unstack('jahr').fillna(0)
piv['Delta_19_24'] = piv[2024] - piv[2019]
piv = piv.sort_values(2024, ascending=False)
print(piv.round(3).to_string())

# Aggregiert nach Invest-Typ
print("\n=== Aggregiert: eigene Invest (UG 81/82) vs. Transfer-Invest (UG 83/86/87/88/89) ===")
inv_agg = (
    d[d['typ_invest'].notna()]
    .groupby(['jahr', 'typ_invest'])['digi_soll_eng']
    .sum().div(1e6).unstack().fillna(0)
)
inv_agg['Sigma_Digital'] = d.groupby('jahr')['digi_soll_eng'].sum().div(1e6)
inv_agg['eigene_%'] = (inv_agg['A_eigene_Invest'] / inv_agg['Sigma_Digital'] * 100).round(1)
inv_agg['transfer_%'] = (inv_agg['B_Transfer_Invest'] / inv_agg['Sigma_Digital'] * 100).round(1)
print(inv_agg.round(2).to_string())


=== HG 8 Detail: eigene vs. Transfer-Investitionen (Mrd) ===
jahr   2019   2021   2023   2024  Delta_19_24
ug                                           
89    0.861  2.443  3.425  4.164        3.303
88    0.302  0.194  0.287  1.528        1.226
81    0.927  1.419  1.239  1.084        0.157
83    0.191  0.251  0.048  0.020       -0.171

=== Aggregiert: eigene Invest (UG 81/82) vs. Transfer-Invest (UG 83/86/87/88/89) ===
typ_invest  A_eigene_Invest  B_Transfer_Invest  Sigma_Digital  eigene_%  transfer_%
jahr                                                                               
2019                   0.93               1.35           8.51      10.9        15.9
2021                   1.42               2.89          16.55       8.6        17.5
2023                   1.24               3.76          19.19       6.5        19.6
2024                   1.08               5.71          17.94       6.0        31.8


**Befund.** Die Antwort ist eindeutig: Eigene Bundes-Investitionen (`A_eigene_Invest`, UG 81/82) wachsen absolut nur leicht und schrumpfen anteilig am Digital-Soll. Investitions-Zuschüsse an Dritte (`B_Transfer_Invest`) wachsen dagegen kräftig, absolut wie anteilig, mit UG 89 (Investitionszuschüsse an Sonstige im Inland) als stärkstem Einzeltreiber. Der vermeintliche „Investitions-Sprung" von oben ist fast vollständig ein Transfer-Sprung.

Das ist keine Investitionsoffensive des Bundes. Es ist eine Investitionsoffensive **durch** den Bund **für** andere Akteure. Nächster Schritt: Welche Ressorts und welche konkreten Förderprogramme treiben diesen Transfer-Boom?

In [5]:
#| label: tbl-treiber
print("\n=== Top-Treiber des Transfer-Booms (UG 88+89, Delta 2019->2024 Mrd) ===")
tr = d[d['ug'].isin(['88', '89'])]
piv2 = (
    tr.groupby(['einzelplan', 'einzelplantext', 'jahr'])['digi_soll_eng']
    .sum().div(1e6).unstack('jahr').fillna(0)
)
piv2['Delta_19_24'] = piv2[2024] - piv2[2019]
print(piv2.sort_values('Delta_19_24', ascending=False).head(6).round(2).to_string())


=== Top-Treiber des Transfer-Booms (UG 88+89, Delta 2019->2024 Mrd) ===
jahr                                                                    2019  2021  2023  2024  Delta_19_24
einzelplan einzelplantext                                                                                  
12         Bundesministerium für Digitales und Verkehr                  0.00  0.00  1.68  3.31         3.31
30         Bundesministerium für Bildung und Forschung                  0.37  0.26  0.39  1.72         1.35
6          Bundesministerium des Innern und für Heimat                  0.00  0.00  0.29  0.25         0.25
9          Bundesministerium für Wirtschaft und Klimaschutz             0.00  0.00  1.18  0.25         0.25
25         Bundesministerium für Wohnen, Stadtentwicklung und Bauwesen  0.00  0.00  0.13  0.13         0.13
8          Bundesministerium der Finanzen                               0.00  0.00  0.01  0.01         0.01


**Befund.** Zwei Einzelpläne erklären gemeinsam den Großteil des Anstiegs: EP 12, das Bundesministerium für Digitales und Verkehr (BMDV), und EP 30, das Bundesministerium für Bildung und Forschung (BMBF). Beim BMDV stehen die Breitbandausbau-Förderung an Länder, Kommunen und private Telekommunikationsunternehmen sowie die Förderung der ETCS-Schienenwege dahinter, beides Investitions-Zuschüsse an Dritte, keine eigenen Bundes-Bauten. Beim BMBF treibt die DigitalPakt-Folge (Zuweisungen an Länder für digitale Bildung) den Anstieg, strukturell ein Transfer, der über die Schulträger weiter verteilt wird.

[TODO: Externe Validierung über IT-Großprojekte-Bericht und Bundesrechnungshof-Bemerkungen für die BMDV-Posten Breitbandausbau und ETCS.]

[TODO: Externe Validierung über KMK-Berichte für die BMBF-Position DigitalPakt-Folge.]

Nächster Schritt: Hält dieses Muster auch unter der weiten Digital-Abgrenzung, oder ist es ein Artefakt der engen Abgrenzung?

In [6]:
print("\n=== Robustheit: gleiche Struktur unter digi_soll_weit ===")
dw = df[(df['any_tag'] == 1) & (df['digi_soll_weit'] > 0)].copy()
dw['typ_fein'] = dw['ug'].map(
    lambda x: 'A_eigene' if x in ['81', '82']
    else ('B_transfer' if x in ['83', '86', '87', '88', '89']
          else ('konsumtiv' if x[:1] in ['4', '5', '6'] else 'sonstiges'))
)
agg_w = dw.groupby(['jahr', 'typ_fein'])['digi_soll_weit'].sum().div(1e6).unstack().round(2)
print(agg_w.to_string())


=== Robustheit: gleiche Struktur unter digi_soll_weit ===
typ_fein  A_eigene  B_transfer  konsumtiv  sonstiges
jahr                                                
2019          0.93        1.45       7.21       0.04
2021          1.43        3.00      13.33       0.03
2023          1.24        3.89      15.38       0.04
2024          1.09        5.86      12.09       0.05


**Befund.** Unter der weiten Abgrenzung (`digi_soll_weit`) zeigt sich dieselbe Struktur: Eigene Investitionen bleiben nahezu konstant, Transfer-Investitionen wachsen deutlich. Das Muster ist kein Artefakt der engen Abgrenzung, sondern robust, die Schere zwischen Eigenleistung und Verteilung ist ein stabiles Merkmal, kein Abgrenzungseffekt.

Nächster Schritt: Ein Spin-off-Blick auf den auffälligen HG-6-Peak 2023, was steckt dahinter?

In [7]:
print("\n=== HG-6-Peak 2023: Was steckt dahinter? ===")
hg6 = d[d['hg'] == '6'].copy()
# Mikroelektronik-Titel isolieren
is_mikro = (
    (hg6['einzelplan'] == 60)
    & (hg6['gruppe'] == 686)
    & (hg6['titel_text'].fillna('').str.contains('Mikroelektronik', case=False))
)
print(f"Mikroelektronik-Titel (EP 60, Gruppe 686): {is_mikro.sum()} Zeilen")
print("Volumen je Jahr (Mrd):")
print(hg6[is_mikro].groupby('jahr')['digi_soll_eng'].sum().div(1e6).round(3).to_string())

print("\nHG 6 mit/ohne Mikroelektronik-Titel (Mrd):")
total_hg6 = hg6.groupby('jahr')['digi_soll_eng'].sum().div(1e6)
ohne = hg6[~is_mikro].groupby('jahr')['digi_soll_eng'].sum().div(1e6)
out = pd.DataFrame({'HG6_total': total_hg6, 'HG6_ohne_Mikro': ohne}).round(2)
print(out.to_string())


=== HG-6-Peak 2023: Was steckt dahinter? ===
Mikroelektronik-Titel (EP 60, Gruppe 686): 1 Zeilen
Volumen je Jahr (Mrd):
jahr
2023    2.74

HG 6 mit/ohne Mikroelektronik-Titel (Mrd):
      HG6_total  HG6_ohne_Mikro
jahr                           
2019       2.61            2.61
2021       5.99            5.99
2023       7.97            5.23
2024       4.60            4.60


**Befund.** Ein einziger Titel erklärt einen erheblichen Teil des HG-6-Peaks 2023: die Mikroelektronik-Förderung in EP 60 (Allgemeine Finanzverwaltung, Gruppe 686). Ohne diesen Titel liegt das HG-6-Volumen 2023 deutlich näher an der Trendlinie der Nachbarjahre. Das ist ein Einzeleffekt und der Übergang zum nächsten Thema: Wofür wird EP 60 eigentlich genutzt?

## Spin-off: EP 60 als Schatten-Förderkanal

Einzelplan 60, die „Allgemeine Finanzverwaltung", bündelt traditionell Ausgaben, die nicht eindeutig einem Fachressort zuzuordnen sind. Seit 2021 wird er systematisch für **digitale Industriepolitik im großen Stil** genutzt, der oben gefundene Mikroelektronik-Titel ist nur die Spitze:

- **2023:** Mikroelektronik-Förderung mit einem Sollansatz von 2,74 Mrd. €. Hintergrund: deutsche Beteiligung an der EU-Chips-Act-Initiative, konkret an Intel-Magdeburg und ESMC-Dresden. [TODO: Validierung über Bundestags-Drucksachen zum Chips-Act.]
- **2021:** Je rund 0,4 Mrd. € für Künstliche Intelligenz und Quantentechnologie.
- **2024:** Praktisch null, nicht „gestoppt", sondern als mehrjährige Verpflichtungsermächtigung im Bewilligungsjahr ausgewiesen (siehe Methoden-Box unten).

**Politische Implikation.** Diese Mittel sind klassifikatorisch Teil des Digitalhaushalts, inhaltlich aber überwiegend Industriepolitik in digital geprägten Branchen. Eine zentrale Steuerung durch ein Bundesministerium für Digitales und Staatsmodernisierung (BMDS) würde hier die Industriepolitik-Hoheit der Fachressorts berühren, des Bundesministeriums für Wirtschaft und Klimaschutz (BMWK) und des BMBF, eine weitere, in Notebook 01 nicht erfasste Grenze der Bündelungsreichweite.

## Methoden-Box: Verpflichtungsermächtigungen verzerren Soll-Trends

Sieben der acht größten Soll-Rückgänge von 2023 auf 2024 in der Hauptgruppe 6 (laufende Zuweisungen) enthalten im Titeltext das Wort „Verpflichtungsermächtigung", das erklärt einen Teil des oben sichtbaren HG-6-Rückgangs mit.

**Was ist eine Verpflichtungsermächtigung (VE)?** Eine mehrjährige Förderzusage, die im Bewilligungsjahr als hoher Sollansatz erscheint und in den Folgejahren als Null. Der Politikinhalt läuft im Hintergrund weiter, nur die Soll-Reihe sieht aus, als sei das Programm gestoppt worden.

**Methodische Konsequenz.** Wer Soll-Zeitreihen interpretiert, wie wir es in diesem Notebook durchgehend tun, muss diesen Mechanismus mitdenken. Ein Soll-Rückgang ist nicht automatisch ein Politikwechsel; er kann ebenso gut eine Verschiebung der haushaltstechnischen Verbuchung sein. Das gilt auch für das EP-60-Muster im folgenden Abschnitt: Das Verschwinden eines Postens 2024 heißt nicht, dass das Programm beendet wurde.

## Kernbefund und weiter

**Eigene Bundes-Investitionen in Digitalisierung stagnieren absolut bei rund einer Milliarde Euro und schrumpfen anteilig; Investitions-Zuschüsse an Länder und Private wachsen kräftig und verdoppeln ihren Anteil am Digital-Soll. Der Bund verteilt zunehmend, baut aber selbst nicht zunehmend auf.** Getrieben wird das vor allem von zwei Förderprogrammen: Breitbandausbau/ETCS (BMDV, EP 12) und DigitalPakt-Folge (BMBF, EP 30).

**Weiter zu Notebook 03** Akt 3: Wie wird darüber geredet? Wenn der Bund überwiegend fördert statt baut, lohnt der Blick auf die Sprache, mit der diese Förderung politisch verkauft wird.

**Limitationen dieser Analyse.** Wir analysieren Sollansätze, nicht Ist-Auszahlungen, ob die Fördermittel tatsächlich abfließen und wirken, ist eine andere Frage. Verpflichtungsermächtigungen (siehe Methoden-Box oben) können Jahresvergleiche verzerren. Die drei großen Transfer-Posten (Breitband, ETCS, DigitalPakt-Folge) sind bislang nur datenseitig identifiziert, die externe Validierung steht aus.